In [ ]:
#!pip install pandas numpy faker openai tqdm
#!pip install pandas numpy faker openai tqdm openpyxl

In [ ]:
import os
print(os.environ.get("OPENAI_API_KEY"))
myOpenAIKey = ''

None


In [11]:
import pandas as pd
import numpy as np
from faker import Faker
from openai import OpenAI
from tqdm import tqdm
import os
import time
from datetime import timedelta

# Configuración inicial
N_RECORDS = 5000
FRAUD_RATE = 0.10
NOISE_RATE = 0.15
OUTPUT_FILE = "dataset_reclamos_ia_ruidoso_extremo.xlsx"

fake = Faker('es_ES')
# Recuerda usar variables de entorno por seguridad
client = OpenAI(api_key=myOpenAIKey)

def generar_datos_tabulares(n):
    data = []
    
    makes_models = {
        'Honda': ['Civic', 'Accord', 'CR-V', 'Fit', 'HR-V', 'Pilot'],
        'Hyundai': ['Sonata', 'Tucson', 'Elantra', 'Santa Fe', 'Kona', 'Ioniq 5'],
        'Kia': ['K5', 'Sportage', 'Sorento', 'Rio', 'EV6', 'Telluride'],
        'Toyota': ['Corolla', 'Camry', 'Hilux', 'RAV4', 'Prius', 'Yaris', 'Land Cruiser'],
        'Ford': ['Escape', 'Explorer', 'Ranger', 'F-150', 'Mustang', 'Bronco'],
        'Chevrolet': ['Silverado', 'Equinox', 'Malibu', 'Tahoe', 'Camaro', 'Cruze'],
        'Nissan': ['Sentra', 'Altima', 'Rogue', 'Frontier', 'Versa', 'Pathfinder'],
        'Volkswagen': ['Jetta', 'Golf', 'Tiguan', 'Passat', 'Polo', 'Amarok'],
        'BMW': ['Series 3', 'Series 5', 'X3', 'X5', 'M4', 'iX'],
        'Mercedes-Benz': ['C-Class', 'E-Class', 'S-Class', 'GLE', 'GLC', 'EQS'],
        'Audi': ['A3', 'A4', 'A6', 'Q3', 'Q5', 'e-tron'],
        'Mazda': ['Mazda3', 'Mazda6', 'CX-5', 'CX-30', 'MX-5 Miata'],
        'Subaru': ['Impreza', 'Forester', 'Outback', 'Crosstrek', 'WRX'],
        'Lexus': ['RX', 'NX', 'ES', 'IS', 'GX'],
        'Tesla': ['Model 3', 'Model Y', 'Model S', 'Model X', 'Cybertruck'],
        'Volvo': ['XC40', 'XC60', 'XC90', 'S60', 'V60'],
        'Jeep': ['Wrangler', 'Grand Cherokee', 'Renegade', 'Compass', 'Gladiator'],
        'Dodge': ['Challenger', 'Charger', 'Durango', 'Hornet'],
        'Ram': ['1500', '2500', '700', 'Vans'],
        'Land Rover': ['Range Rover', 'Defender', 'Discovery', 'Evoque'],
        'Jaguar': ['F-Type', 'F-Pace', 'XE', 'I-Pace'],
        'Porsche': ['911', 'Cayenne', 'Macan', 'Taycan', 'Panamera'],
        'Ferrari': ['SF90', 'F8', 'Roma', 'Purosangue'],
        'Lamborghini': ['Aventador', 'Huracan', 'Urus', 'Revuelto'],
        'Mitsubishi': ['Lancer', 'Outlander', 'Montero', 'L200', 'ASX'],
        'Suzuki': ['Swift', 'Jimny', 'Vitara', 'Baleno', 'Ignis'],
        'Peugeot': ['208', '3008', '5008', '2008', 'Partner'],
        'Renault': ['Clio', 'Kwid', 'Duster', 'Sandero', 'Megane'],
        'Fiat': ['500', 'Cronos', 'Pulse', 'Toro', 'Argo'],
        'Chrysler': ['300', 'Pacifica', 'Voyager'],
        'Cadillac': ['Escalade', 'CT4', 'CT5', 'Lyriq'],
        'Buick': ['Encore', 'Enclave', 'Envision'],
        'GMC': ['Sierra', 'Yukon', 'Terrain', 'Canyon', 'Hummer EV'],
        'Acura': ['TLX', 'MDX', 'RDX', 'Integra'],
        'Infiniti': ['Q50', 'QX60', 'QX80', 'QX50'],
        'Genesis': ['G70', 'G80', 'GV70', 'GV80'],
        'Alfa Romeo': ['Giulia', 'Stelvio', 'Tonale'],
        'Maserati': ['Ghibli', 'Levante', 'Quattroporte', 'Grecale'],
        'Aston Martin': ['DB11', 'Vantage', 'DBS', 'DBX'],
        'Bentley': ['Continental GT', 'Bentayga', 'Flying Spur'],
        'Rolls-Royce': ['Phantom', 'Ghost', 'Cullinan', 'Spectre'],
        'Mini': ['Cooper', 'Countryman', 'Clubman'],
        'Skoda': ['Octavia', 'Fabia', 'Kodiaq', 'Superb'],
        'Seat': ['Ibiza', 'Leon', 'Ateca', 'Tarraco'],
        'Cupra': ['Formentor', 'Leon', 'Born'],
        'BYD': ['Han', 'Tang', 'Atto 3', 'Dolphin'],
        'MG': ['MG4', 'ZS', 'HS', 'Marvel R'],
        'Chery': ['Tiggo 2', 'Tiggo 7', 'Tiggo 8'],
        'Great Wall': ['Poer', 'Haval H6', 'Jolion'],
        'Rivian': ['R1T', 'R1S']
    }
    workshops = ['Taller Autorizado Principal', 'Auto Pintura J&J', 'Mecánica General Los Mina', 'Centro Automotriz Naco', 'Taller Los Prados', 'Desabolladura El Maestro']
    branches = ['Oficina Principal', 'Sucursal Santiago', 'Sucursal Zona Oriental', 'Punta Cana', 'Sucursal La Vega']
    loss_types = ['Pérdida Parcial', 'Pérdida Total']
    coverages = ['Daños Propios', 'Responsabilidad Civil', 'Cobertura Completa', 'Pérdida Total', 'Colisión', 'Robo', 'Vandalismo', 'Incendio', 'Inundación', 'Rotura de Cristal']
    occupations = ['Ingeniero', 'Comerciante', 'Abogado', 'Desempleado', 'Estudiante', 'Médico', 'Empleado Privado', 'Independiente']
    zips = ['10101', '11503', '51000', '11501', '10514', '10509', '10508', '10580', '10520', '11500', '51100', '51200']

    for _ in range(n):
        es_fraude = np.random.choice([0, 1], p=[1 - FRAUD_RATE, FRAUD_RATE])
        
        incident_date = fake.date_time_between(start_date='-1y', end_date='now')
        
        if es_fraude == 1:
            days_to_report = np.random.randint(5, 30)
            policy_start_date = incident_date - timedelta(days=np.random.randint(1, 15))
        else:
            days_to_report = np.random.randint(0, 3)
            policy_start_date = incident_date - timedelta(days=np.random.randint(30, 1000))
            
        date_reported = incident_date + timedelta(days=days_to_report)
        insured_inception_date = policy_start_date - timedelta(days=np.random.randint(0, 2000))
        last_purchase_date = policy_start_date - timedelta(days=np.random.randint(0, 365))
        policy_renewal_date = policy_start_date + timedelta(days=365)

        make = np.random.choice(list(makes_models.keys()))
        model = np.random.choice(makes_models[make])
        model_year = np.random.randint(2005, 2025)
        
        coverage_amount = np.round(np.random.uniform(300000, 2500000), 2)
        premium_amount = np.round(coverage_amount * np.random.uniform(0.02, 0.08), 2)
        
        if es_fraude == 1:
            claim_amount = np.round(coverage_amount * np.random.uniform(0.8, 0.98), 2)
            history_count = np.random.randint(2, 6)
            freq_12_months = np.random.randint(2, 5)
            workshop = np.random.choice(['Auto Pintura J&J', 'Mecánica General Los Mina'])
        else:
            claim_amount = np.round(coverage_amount * np.random.uniform(0.05, 0.4), 2)
            history_count = np.random.randint(0, 2)
            freq_12_months = np.random.randint(0, 2)
            workshop = np.random.choice(workshops)

        data.append({
            "Customer_Age": np.random.randint(18, 85),
            "Claim_Amount": claim_amount,
            "Claim_History_Count_This_Policy": history_count,
            "Claim_Frequency_Last_12_Month": freq_12_months,
            "Gender": np.random.choice(["M", "F"]),
            "Last_Purchase_History_Date": last_purchase_date.strftime("%Y-%m-%d"),
            "Policy_Start_Date": policy_start_date.strftime("%Y-%m-%d"),
            "Policy_Renewal_Date": policy_renewal_date.strftime("%Y-%m-%d"),
            "Coverage_Amount": coverage_amount,
            "Premium_Amount": premium_amount,
            "Incident_Date": incident_date.strftime("%Y-%m-%d %H:%M"),
            "Date_Reported": date_reported.strftime("%Y-%m-%d %H:%M"),
            "Insured_Inception_Date": insured_inception_date.strftime("%Y-%m-%d"),
            "Insured_MaritalStatus": np.random.choice(["Soltero", "Casado", "Divorciado", "Viudo"]),
            "Insured_Occupation": np.random.choice(occupations),
            "Coverage_description": np.random.choice(coverages),
            "LossType_Description": np.random.choice(loss_types),
            "Branch_Description": np.random.choice(branches),
            "Beneficiary_Type_Description": np.random.choice(['Asegurado', 'Tercero', 'Banco/Financiera']),
            "WorkShop_Name": workshop,
            "Insured_Zip": np.random.choice(zips),
            "Vehicle_Make": make,
            "Vehicle_Model": model,
            "Model_Year": model_year,
            "Prediccion_Fraude": es_fraude,
            "Claim_Description": ""
        })
        
    return pd.DataFrame(data)

def inyectar_ruido(df):
    """
    Introduce imperfecciones lógicas, numéricas y de fechas, 
    manteniendo la limpieza de los textos (sin nulos ni minúsculas).
    """
    print("Inyectando ruido lógico y financiero en el dataset...")
    df_noisy = df.copy()
    n_rows = len(df_noisy)
    
    # 1. Ruido en Etiqueta Objetivo
    mask_label = np.random.rand(n_rows) < (NOISE_RATE / 2)
    df_noisy.loc[mask_label, 'Prediccion_Fraude'] = 1 - df_noisy.loc[mask_label, 'Prediccion_Fraude']

    # 2. Ruido en Fechas (Ej: Date_Reported anterior a Incident_Date)
    mask_dates = np.random.rand(n_rows) < (NOISE_RATE / 2)
    temp_dates = pd.to_datetime(df_noisy.loc[mask_dates, 'Incident_Date'])
    df_noisy.loc[mask_dates, 'Date_Reported'] = (temp_dates - pd.to_timedelta(np.random.randint(1, 10, size=mask_dates.sum()), unit='d')).dt.strftime('%Y-%m-%d %H:%M')

    # 3. Ruido Numérico y Financiero
    mask_claim = np.random.rand(n_rows) < NOISE_RATE
    multipliers = np.random.choice([10, -1, 0.1], size=mask_claim.sum())
    # Multiplicamos el array de la columna por el array de multiplicadores
    df_noisy.loc[mask_claim, 'Claim_Amount'] = df_noisy.loc[mask_claim, 'Claim_Amount'].values * multipliers

    # Edades imposibles (Outliers)
    #mask_age = np.random.rand(n_rows) < NOISE_RATE
    #df_noisy.loc[mask_age, 'Customer_Age'] = np.random.choice([0, 15, 120, 200], size=mask_age.sum())

    # Años de vehículos imposibles
    #mask_year = np.random.rand(n_rows) < NOISE_RATE
    #df_noisy.loc[mask_year, 'Model_Year'] = np.random.choice([1900, 2050, 0], size=mask_year.sum())

    return df_noisy

def generar_texto_siniestro(row):
    try:
        incident_dt = pd.to_datetime(row['Incident_Date'])
        reported_dt = pd.to_datetime(row['Date_Reported'])
        dias_retraso = (reported_dt - incident_dt).days
    except:
        dias_retraso = "Desconocido"

    contexto = f"""
    Tipo de Pérdida: {row['LossType_Description']}
    Monto Reclamado: RD${row['Claim_Amount']}
    Retraso en el reporte: {dias_retraso} días
    Edad Asegurado: {row['Customer_Age']} años
    """

    if row['Prediccion_Fraude'] == 0:
        # Prompt para casos Legítimos (Basado en la literatura: específicos, emocionales, directos)
        instruccion = (
            "Eres un cliente dominicano reportando un siniestro de vehículo legítimo a tu aseguradora. "
            "Escribe como si estuvieras llenando el reporte desde tu celular. "
            "Sé directo y específico sobre lo que pasó en el accidente. Muestra una profundidad emocional normal "
            "(ej. un poco de frustración, susto o molestia por el choque). "
            "No te justifiques de más ni des detalles previos al accidente que no importen. "
            "Comete al menos dos errores de ortografía comunes (ej. b por v, omitir h, falta de tildes). "
            "Ignora algunos puntos, pero sé breve."
            "No incluyas el monto reclamado, solo lo paso en el contexto para que tengas mas informacion sobre el reclamo y te ayude"
            "No incluyas la edad del asegurado, solo lo paso en el contexto para que tengas mas informacion sobre el asegurado"
        )
    else:
        # Prompt para casos de Fraude (Basado en el paper SA4: distanciamiento, vaguedad, superlativos)
        instruccion = (
            "Eres un estafador en República Dominicana intentando cobrar un seguro de auto. "
            "Debes redactar el siniestro usando marcadores lingüísticos de engaño comprobados: "
            "1. 'Distanciamiento emocional': usa voz pasiva para alejar tu culpa (ej. 'el vehículo resultó impactado' en vez de 'choqué'). "
            "2. 'Falta de especificidad': sé muy vago sobre el momento exacto del impacto, pero da demasiados detalles irrelevantes "
            "sobre a dónde ibas o qué hacías antes. "
            "3. 'Uso excesivo de superlativos': usa palabras como 'absolutamente', 'completamente', 'perfectamente' para tratar de sonar convincente. "
            "Exige que te paguen rápido. Comete algunos errores gramaticales por escribir rápido."
            "No incluyas el monto reclamado, solo lo paso en el contexto para que tengas mas informacion sobre el reclamo y te ayude"
            "No incluyas la edad del asegurado, solo lo paso en el contexto para que tengas mas informacion sobre el asegurado"
        )

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": instruccion},
                {"role": "user", "content": f"Escribe la descripción del siniestro basándote en este contexto (máximo 4 oraciones):\n{contexto}"}
            ],
            temperature=0.85, # Ligeramente más alto para mayor variabilidad en las excusas
            max_tokens=150
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return "ERROR_API"
        
print(f"1. Generando base tabular perfecta de {N_RECORDS} registros...")
df_perfecto = generar_datos_tabulares(N_RECORDS)

print("2. Aplicando inyección de ruido (fechas, lógica y outliers)...")
df = inyectar_ruido(df_perfecto)

print("3. Generando Claim_Descriptions con ruido lingüístico (OpenAI)...")
descripciones = []

for index, row in tqdm(df.iterrows(), total=df.shape[0]):
    texto = generar_texto_siniestro(row)
    descripciones.append(texto)
    time.sleep(0.05) 
    
    if (index + 1) % 500 == 0:
        df_temp = df.iloc[:index+1].copy()
        df_temp['Claim_Description'] = descripciones[:index+1]
        df_temp.to_excel(f"backup_temp_{index+1}.xlsx", index=False, engine='openpyxl')

df['Claim_Description'] = descripciones

columnas_ordenadas = [
    'Customer_Age', 'Claim_Amount', 'Claim_History_Count_This_Policy', 'Claim_Frequency_Last_12_Month', 
    'Claim_Description', 'Gender', 'Last_Purchase_History_Date', 'Policy_Start_Date', 'Policy_Renewal_Date', 
    'Coverage_Amount', 'Premium_Amount', 'Incident_Date', 'Date_Reported', 'Insured_Inception_Date', 
    'Insured_MaritalStatus', 'Insured_Occupation', 'Coverage_description', 'LossType_Description', 
    'Branch_Description', 'Beneficiary_Type_Description', 'WorkShop_Name', 'Insured_Zip', 'Vehicle_Make', 
    'Vehicle_Model', 'Model_Year', 'Prediccion_Fraude'
]

df = df[columnas_ordenadas]

print(f"4. Guardando dataset ruidoso en {OUTPUT_FILE}...")
df.to_excel(OUTPUT_FILE, index=False, engine='openpyxl')
print("¡Completado!")

1. Generando base tabular perfecta de 5000 registros...
2. Aplicando inyección de ruido (fechas, lógica y outliers)...
Inyectando ruido lógico y financiero en el dataset...
3. Generando Claim_Descriptions con ruido lingüístico (OpenAI)...


100%|███████████████████████████████████████| 5000/5000 [2:59:50<00:00,  2.16s/it]


4. Guardando dataset ruidoso en dataset_reclamos_ia_ruidoso_extremo.xlsx...
¡Completado!
